# C15_E02 — Modelo híbrido físico-datos

**Capítulo:** 15 — Gemelos digitales, modelos híbridos y simulación conectada  
**Identificador:** `C15_E02`  
**Objetivo pedagógico:** Combinar modelo físico y corrección residual con ML manteniendo interpretabilidad.  
**Librerías:** numpy, scikit-learn, matplotlib

## Ejemplo industrial

Modelo de proceso con física parcialmente conocida; residuo capturado por ML.

---

> **Aviso de uso responsable.** Este notebook es didáctico. No es código de producción. Cualquier implementación real requiere validación independiente. Ver `docs/politica_uso_responsable.md`.

## 1. Modelo híbrido: físico + corrección residual ML

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import os
np.random.seed(0)

# Modelo físico simplificado: y_phys = K*u
K_phys = 1.5
N = 200
u = np.random.uniform(0, 5, N)

# Realidad: y_real = K*u + g(u) + ruido (g = no linealidad cuadrática que el físico no captura)
y_real = K_phys*u + 0.3*u**2 + 0.1*np.random.randn(N)

# Modelo físico solo
y_phys = K_phys*u
# Residuo = realidad − físico
residuo = y_real - y_phys

# Aprender el residuo en función de u
reg = LinearRegression().fit(u.reshape(-1,1)**2, residuo)
print("Pendiente del residuo aprendido (esperado ≈ 0.3):", reg.coef_[0])

# Modelo híbrido = físico + residuo aprendido
y_hibrido = y_phys + reg.predict(u.reshape(-1,1)**2)

Pendiente del residuo aprendido (esperado ≈ 0.3): 0.29837611723443375


## 2. Visualización

In [2]:
order = np.argsort(u)
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(u, y_real, alpha=0.5, label="Realidad")
ax.plot(u[order], y_phys[order], label="Modelo físico", color='C1')
ax.plot(u[order], y_hibrido[order], '--', label="Modelo híbrido", color='C2')
ax.set_xlabel("u"); ax.set_ylabel("y"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("C15_E02 — Modelo híbrido físico-datos")
out = '../../figures/cap15/fig_C15_F03_hibrido.png'
os.makedirs(os.path.dirname(out), exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=120); plt.show()

## 3. Interpretación

El modelo físico `y = K·u` captura la tendencia lineal pero pierde la curvatura. El modelo híbrido aprende el residuo cuadrático con regresión lineal y lo suma al físico, mejorando el ajuste. **Ventaja:** la parte física se mantiene interpretable. **Riesgo:** el término data-driven puede absorber errores físicos de modelado y dar falsa sensación de calibración. Mantener el término físico tan completo como sea posible.